<a href="https://colab.research.google.com/github/yashita-mishra/AI-Project-Child-Nutrition/blob/main/Nhanescleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import urllib.request

import numpy as np
import pandas as pd

BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/"
DATA_DIR = "data"

KEEP = {
    "DEMO_L": [
        "SEQN", "RIDSTATR", "RIAGENDR", "RIDAGEYR", "RIDAGEMN", "RIDEXAGM",
        "RIDRETH3", "DMDBORN4", "DMDHHSIZ", "DMDHRGND", "DMDHRAGZ", "DMDHREDZ",
        "DMDHSEDZ", "DMDHRMAZ", "INDFMPIR", "WTINT2YR", "WTMEC2YR",
        "SDMVSTRA", "SDMVPSU",
    ],
    "BMX_L": ["SEQN", "BMXWT", "BMXHT", "BMXBMI", "BMXRECUM"],
    "CBC_L": ["SEQN", "LBXHGB", "LBXHCT"],
    "ECQ_L": ["SEQN", "ECD070A", "ECD070B", "ECQ080", "ECQ090", "WHQ030E", "MCQ080E"],
    "DBQ_L": ["SEQN", "DBQ010", "DBD030", "DBD041", "DBD050", "DBD055", "DBD061"],
    "FSQ_L": [
        "SEQN", "FSDAD", "FSD162", "FSQ012", "FSQ165", "FSQ653", "FSD660ZC",
        "FSD675", "FSD680", "FSD670ZC", "FSQ690",
    ],
}

MISSING_CODES = {
    **{c: [7, 9] for c in [
        "DBQ010", "ECQ080", "ECQ090", "WHQ030E", "MCQ080E", "FSD162", "FSQ012",
        "FSQ165", "FSQ653", "FSD660ZC", "FSD675", "FSD680", "FSQ690",
    ]},
    "FSD670ZC": [777, 999],
    "ECD070A": [7777, 9999],
    "ECD070B": [7777, 9999],
    **{c: [777777, 999999] for c in ["DBD030", "DBD041", "DBD050", "DBD055", "DBD061"]},
}

CENSORED = {"DBD030": 1096, "DBD041": 366, "DBD050": 1096, "DBD055": 731, "DBD061": 1096}

ZERO_MEANS = {
    "DBD030": "still_breastfeeding",
    "DBD041": "formula_never",
    "DBD050": "still_formula",
    "DBD055": "other_food_never",
    "DBD061": "milk_never",
}


def fetch(name: str) -> pd.DataFrame:
    path = os.path.join(DATA_DIR, f"{name}.xpt")
    if not os.path.exists(path):
        os.makedirs(DATA_DIR, exist_ok=True)
        print(f"downloading {name} ...")
        req = urllib.request.Request(
            BASE + name + ".xpt", headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(req) as r, open(path, "wb") as f:
            f.write(r.read())
    df = pd.read_sas(path, format="xport")
    keep = [c for c in KEEP[name] if c in df.columns]
    absent = sorted(set(KEEP[name]) - set(keep))
    if absent:
        print(f"[{name}] expected columns not found: {absent}")
    return df[keep]

In [2]:
frames = {name: fetch(name) for name in KEEP}

df = frames["DEMO_L"]
n_demo = len(df)
for name, part in frames.items():
    if name == "DEMO_L":
        continue
    df = df.merge(part, on="SEQN", how="left", validate="one_to_one")
    assert len(df) == n_demo, f"{name} changed the row count"
print(f"merged table: {len(df)} rows x {df.shape[1]} columns")

# refused / don't know -> NaN
for col, codes in MISSING_CODES.items():
    if col in df:
        df[col] = df[col].replace(codes, np.nan)

# censored ages: flag, then replace by the cap
for col, cap in CENSORED.items():
    if col in df:
        df[f"{col}_censored"] = df[col] == 666666
        df[col] = df[col].where(df[col] != 666666, cap)

# zeros meaning "still" / "never": explicit flags, blank the age
for col, flag in ZERO_MEANS.items():
    if col in df:
        df[flag] = df[col] == 0
        df[col] = df[col].replace(0, np.nan)

kids = df[df["RIDAGEYR"].between(2, 6) & (df["RIDSTATR"] == 2)].copy()
print(f"children aged 2-6 who were examined: {len(kids)}")

downloading DEMO_L ...
downloading BMX_L ...
downloading CBC_L ...
downloading ECQ_L ...
downloading DBQ_L ...
downloading FSQ_L ...
merged table: 11933 rows x 47 columns
children aged 2-6 who were examined: 664


In [3]:
if {"ECD070A", "ECD070B"} <= set(kids.columns):
    kids["birth_wt_lb"] = kids["ECD070A"] + kids["ECD070B"] / 16
    lbw = pd.Series(
        np.where(kids["birth_wt_lb"].notna(), kids["birth_wt_lb"] < 5.5, np.nan),
        index=kids.index,
    )
    if "ECQ080" in kids:
        lbw = lbw.fillna(kids["ECQ080"].map({1: 0.0, 2: 1.0}))
    kids["low_birth_weight"] = lbw

if {"DMDHRGND", "DMDHREDZ", "DMDHSEDZ"} <= set(kids.columns):
    kids["parent_edu"] = np.where(
        kids["DMDHRGND"] == 2, kids["DMDHREDZ"], kids["DMDHSEDZ"]
    )

if "LBXHGB" in kids:
    cutoff = np.where(kids["RIDAGEYR"] < 5, 11.0, 11.5)
    kids["anaemia"] = np.where(
        kids["LBXHGB"].notna(), (kids["LBXHGB"] < cutoff).astype(float), np.nan
    )

kids.to_csv(os.path.join(DATA_DIR, "children_2_6.csv"), index=False)
kids.isna().mean().sort_values(ascending=False).round(2).head(25)

,0
RIDAGEMN,0.99
ECQ090,0.98
ECQ080,0.98
BMXRECUM,0.69
FSD680,0.60
FSD670ZC,0.55
FSD675,0.55
FSD660ZC,0.55
anaemia,0.49
LBXHCT,0.49


In [4]:
from google.colab import files
files.download("data/children_2_6.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Re-lable columns: to easily interpretable names

In [1]:
import pandas as pd

RENAME_MAP = {
    "SEQN": "id",
    "RIDSTATR": "exam_status",              # 1 interview only, 2 interviewed + examined
    "RIAGENDR": "sex",                      # 1 male, 2 female
    "RIDAGEYR": "age_years",
    "RIDAGEMN": "age_months_screening",     # mostly empty for ages 2-6
    "RIDEXAGM": "age_months_exam",          # needed for growth z-scores
    "RIDRETH3": "ethnicity",
    "DMDBORN4": "country_of_birth",         # 1 US, 2 elsewhere
    "DMDHHSIZ": "household_size",
    "DMDHRGND": "ref_person_sex",           # 1 male, 2 female
    "DMDHRAGZ": "ref_person_age_group",
    "DMDHREDZ": "ref_person_education",
    "DMDHSEDZ": "ref_spouse_education",
    "DMDHRMAZ": "ref_person_marital_status",
    "INDFMPIR": "income_poverty_ratio",     # 0-5, capped at 5
    "WTINT2YR": "weight_interview",
    "WTMEC2YR": "weight_exam",              # use for exam variables
    "SDMVSTRA": "survey_strata",
    "SDMVPSU": "survey_psu",

    #Body measures (BMX_L)
    "BMXWT": "weight_kg",
    "BMXHT": "height_cm",
    "BMXBMI": "bmi",
    "BMXRECUM": "recumbent_length_cm",

    #Blood count (CBC_L)
    "LBXHGB": "hemoglobin_gdl",
    "LBXHCT": "hematocrit_pct",

    #Early childhood (ECQ_L)
    "ECD070A": "birth_wt_pounds_raw",       # 3 = <=3 lb, 11 = >=11 lb
    "ECD070B": "birth_wt_ounces_raw",
    "ECQ080": "birth_wt_over_5_5lb",        # 1 more, 2 less
    "ECQ090": "birth_wt_over_9lb",          # 1 more, 2 less
    "WHQ030E": "parent_view_child_weight",  # 1 over, 2 under, 3 about right
    "MCQ080E": "doctor_said_overweight",    # 1 yes, 2 no

    #Diet behavior (DBQ_L)
    "DBQ010": "ever_breastfed",             # 1 yes, 2 no
    "DBD030": "breastfeed_stop_days",
    "DBD041": "formula_start_days",
    "DBD050": "formula_stop_days",
    "DBD055": "first_other_food_days",
    "DBD061": "first_milk_days",

    #Food security / programs (FSQ_L)
    "FSDAD": "adult_food_security",         # 1 full ... 4 very low
    "FSD162": "hh_wic_past_year",
    "FSQ012": "hh_snap_past_year",
    "FSQ165": "hh_snap_ever",
    "FSQ653": "child_ever_wic",
    "FSD660ZC": "child_wic_now",
    "FSD675": "child_wic_under_age1",
    "FSD680": "child_wic_age1_to_4",
    "FSD670ZC": "child_wic_months",
    "FSQ690": "mother_wic_pregnancy",

    # "birth_wt_lb": "birth_weight_lb",
    # "low_birth_weight": "low_birth_weight",  # unchanged, already clear
    # "parent_edu": "mother_education",
    # "anaemia": "anaemia",                    # unchanged, already clear
}


def rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Rename columns using RENAME_MAP; warn about any mapped names not in df."""
    missing = [c for c in RENAME_MAP if c not in df.columns]
    if missing:
        print("Not found in data (skipped):", missing)
    return df.rename(columns=RENAME_MAP)


if __name__ == "__main__":
    df = pd.read_csv("children_2_6.csv")
    df = rename_columns(df)
    print(df.columns.tolist())
    df.to_csv("children_2_6_renamed.csv", index=False)

['id', 'exam_status', 'sex', 'age_years', 'age_months_screening', 'age_months_exam', 'ethnicity', 'country_of_birth', 'household_size', 'ref_person_sex', 'ref_person_age_group', 'ref_person_education', 'ref_spouse_education', 'ref_person_marital_status', 'income_poverty_ratio', 'weight_interview', 'weight_exam', 'survey_strata', 'survey_psu', 'weight_kg', 'height_cm', 'bmi', 'recumbent_length_cm', 'hemoglobin_gdl', 'hematocrit_pct', 'birth_wt_pounds_raw', 'birth_wt_ounces_raw', 'birth_wt_over_5_5lb', 'birth_wt_over_9lb', 'parent_view_child_weight', 'doctor_said_overweight', 'ever_breastfed', 'breastfeed_stop_days', 'formula_start_days', 'formula_stop_days', 'first_other_food_days', 'first_milk_days', 'adult_food_security', 'hh_wic_past_year', 'hh_snap_past_year', 'hh_snap_ever', 'child_ever_wic', 'child_wic_now', 'child_wic_under_age1', 'child_wic_age1_to_4', 'child_wic_months', 'mother_wic_pregnancy', 'DBD030_censored', 'DBD041_censored', 'DBD050_censored', 'DBD055_censored', 'DBD061_

#### Data Preprocessing and Cleaning

In [5]:
#check for missing values in the renamed dataset
df = pd.read_csv("children_2_6_renamed.csv")
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 664 entries, 0 to 663
Data columns (total 61 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         664 non-null    float64
 1   exam_status                664 non-null    float64
 2   sex                        664 non-null    float64
 3   age_years                  664 non-null    float64
 4   age_months_screening       4 non-null      float64
 5   age_months_exam            664 non-null    float64
 6   ethnicity                  664 non-null    float64
 7   country_of_birth           664 non-null    float64
 8   household_size             664 non-null    float64
 9   ref_person_sex             661 non-null    float64
 10  ref_person_age_group       664 non-null    float64
 11  ref_person_education       628 non-null    float64
 12  ref_spouse_education       392 non-null    float64
 13  ref_person_marital_status  658 non-null    float64

In [12]:
#check for percent of missing values in the renamed dataset
from IPython.display import display, HTML
df.isna().mean().sort_values(ascending=False).round(2)
# display(HTML(f"""
# <div style =max-height: 400px; overflow-y: 
#     {df.to_html()}
# </div>
# """))

age_months_screening      0.99
birth_wt_over_9lb         0.98
birth_wt_over_5_5lb       0.98
recumbent_length_cm       0.69
child_wic_age1_to_4       0.60
                          ... 
ethnicity                 0.00
age_months_exam           0.00
age_years                 0.00
sex                       0.00
doctor_said_overweight    0.00
Length: 61, dtype: float64

In [33]:
#count of columns with missing values percentage greater than or equal to 0.7
missing_pct = df.isna().mean().sort_values(ascending=False).round(2)

print('Missing value count:', (missing_pct >= 0.65).sum())

Missing value count: 4


In [34]:
#name of the columns with missing values percentage greater than or equal to 0.6
df.isna().mean().sort_values(ascending=False).loc[lambda x: x >= 0.65].index

Index(['age_months_screening', 'birth_wt_over_9lb', 'birth_wt_over_5_5lb',
       'recumbent_length_cm'],
      dtype='object')

In [35]:
#drop columns with missing values percentage greater than or equal to 0.65
df_copy = df.drop(columns=df.isna().mean().sort_values(ascending=False).loc[lambda x: x >= 0.65].index)
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 664 entries, 0 to 663
Data columns (total 57 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         664 non-null    float64
 1   exam_status                664 non-null    float64
 2   sex                        664 non-null    float64
 3   age_years                  664 non-null    float64
 4   age_months_exam            664 non-null    float64
 5   ethnicity                  664 non-null    float64
 6   country_of_birth           664 non-null    float64
 7   household_size             664 non-null    float64
 8   ref_person_sex             661 non-null    float64
 9   ref_person_age_group       664 non-null    float64
 10  ref_person_education       628 non-null    float64
 11  ref_spouse_education       392 non-null    float64
 12  ref_person_marital_status  658 non-null    float64
 13  income_poverty_ratio       575 non-null    float64

In [47]:
#check for rows with missing values in the remaining columns, where the portion of missing values in greater than or equal to 0.65 columns is greater than or equal to 0.5
df_copy.isna().mean(axis=1).sort_values(ascending=False).loc[lambda x: x >= 0.65]

Series([], dtype: float64)

#### To do:

- no rows with missing values greater than 0.65 X 57 = 37 rows exist, hence decide whether to impute or not (can't drop)

- predictions to be done on whether anaemic or not

- predictions on ref_person_education or income_poverty_ratio for values that are missing for a given row 

- check the distribution of the data (is it linear or not?)

- draw the DAG, after checking the correlation of variables

- assign probabilities in teh Bayesian Belief network to aid in prediction/ probability assignment for anaemic children